# 03 - Routing and the flat-index comparison

Two experiments:

1. **Router strategy comparison** on 35 hand-labelled queries. Labels were
   assigned from each question's wording before measuring the router, so
   the numbers can disagree with the implementation.
2. **Branched versus flat**, the controlled comparison the project rests
   on. Identical embeddings, BM25, RRF and evidence budget; only the
   partitioning differs.

In [1]:
import sys
from pathlib import Path

CAPSTONE = Path(r"C:\Users\modij\Downloads\MUJ-DS-23FE10CDS00439\capstone")
sys.path.insert(0, str(CAPSTONE / "src"))

from dotenv import load_dotenv
load_dotenv(CAPSTONE / ".env")

import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 70)

In [2]:
from branched_rag.pipeline import BranchedRAGPipeline
from branched_rag.evaluation import (
    load_eval_queries, evaluate_router, router_errors,
    compare_flat_and_branched, FlatBaseline,
)

pipeline = BranchedRAGPipeline.load(CAPSTONE / 'config.yaml')
queries = load_eval_queries(pipeline.config.paths.eval_queries)
print(f'{len(queries)} labelled queries')
print(pd.Series([q.kind for q in queries]).value_counts().to_string())

C:\Users\modij\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


35 labelled queries
single    25
multi     10


## Clause splitting

The fix that made compound questions route correctly.

In [3]:
from branched_rag.retrieval.router import split_clauses

q = 'Should we launch a budget tablet? Consider demand, rivals and financial risk.'
for clause in split_clauses(q):
    print(' -', clause)

 - Should we launch a budget tablet
 - Consider demand
 - rivals
 - financial risk


In [4]:
plan = pipeline.router.route(q)
print('selected:', plan.selected, f'(cutoff {plan.cutoff:.3f})\n')
for d in plan.decisions:
    mark = 'SEL' if d.selected else '   '
    print(f'{mark} {d.branch:11s} {d.score:.3f}  {d.rationale}')

selected: ['news', 'financials'] (cutoff 0.289)

SEL news        0.526  descriptor similarity 0.27; terms: demand, launch, rivals; driven by "Consider demand"
SEL financials  0.501  descriptor similarity 0.41; terms: financial, risk; driven by "financial risk"
    reviews     0.110  descriptor similarity 0.17; driven by "Should we launch a budget tablet? Consider demand, rivals and financial risk."


The clause `financial risk` is what selects the financials branch. Scoring
the whole question as one vector averages three intents into a point near
none of them, and an earlier version selected only `news`.

## Router strategy comparison

In [5]:
evaluate_router(pipeline, queries)

,strategy,subset,queries,exact_match,precision,recall,f1
0,embedding,all,35,0.657,0.833,1.000,0.890
1,embedding,single,25,0.560,0.780,1.000,0.853
2,embedding,multi,10,0.900,0.967,1.000,0.980
3,lexical,all,35,0.686,0.948,0.862,0.883
4,lexical,single,25,0.920,0.940,0.960,0.947
5,lexical,multi,10,0.100,0.967,0.617,0.723
6,hybrid,all,35,0.829,0.948,0.971,0.947
7,hybrid,single,25,0.880,0.940,1.000,0.960
8,hybrid,multi,10,0.700,0.967,0.900,0.913


The two signals fail in opposite directions. Embedding similarity
generalises to paraphrase but over-selects on narrow questions; keyword
overlap is precise on domain terms but collapses on compound questions.
Combining them beats both, which is the empirical case for the default.

In [6]:
table = evaluate_router(pipeline, queries)
overall = table[table.subset == 'all'].set_index('strategy')
print('exact-set match, all queries')
print(overall['exact_match'].to_string())

exact-set match, all queries
strategy
embedding    0.657
lexical      0.686
hybrid       0.829


## Where the router is wrong

Reported rather than tuned away.

In [7]:
router_errors(pipeline, queries)

,id,query,expected,predicted,missed,spurious
0,r07,Which product defects come up repeatedly in user reviews?,reviews,"news,reviews",-,news
1,f03,What do analysts rate the stock and why?,financials,"financials,news",-,news
2,f04,Which investors took stakes or increased their shareholdings?,financials,"financials,news",-,news
3,m01,"Is there consumer demand for budget devices, and what are riva","news,reviews",news,reviews,-
4,m05,How are semiconductor and storage revenues trending in the mar,"financials,news",news,financials,-
5,m06,Do product quality complaints line up with reported margin pre,"financials,reviews","financials,news,reviews",-,news


## Branched versus flat index

The flat baseline pools every chunk into one index, reusing the vectors
read back out of the per-branch FAISS indexes so both systems are
compared on byte-identical embeddings. Each gets the same evidence budget.

In [8]:
baseline = FlatBaseline.from_store(pipeline.store, pipeline.config)
print(f'flat index: {len(baseline.chunks):,} chunks pooled')

flat index: 16,641 chunks pooled


In [9]:
detail, summary = compare_flat_and_branched(pipeline, queries, baseline)
summary

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,kind,queries,branched_coverage,flat_coverage,branched_sources,flat_sources,flat_top_branch_share
0,single,25,1.0,0.880,1.12,1.44,0.912
1,multi,10,0.9,0.633,2.20,1.70,0.818


`flat_top_branch_share` is the fraction of the flat system's evidence
coming from its single most-represented branch. On multi-source questions
the pooled index concentrates most of its evidence in one branch and
covers fewer of the sources the question actually needs.

In [10]:
multi = detail[detail.kind == 'multi']
print('multi-source queries only\n')
print(f"branched coverage : {multi.branched_coverage.mean():.3f}")
print(f"flat coverage     : {multi.flat_coverage.mean():.3f}")
print(f"branched sources  : {multi.branched_sources.mean():.2f}")
print(f"flat sources      : {multi.flat_sources.mean():.2f}")
print(f"flat concentration: {multi.flat_top_branch_share.mean():.1%} from one branch")

multi-source queries only

branched coverage : 0.900
flat coverage     : 0.633
branched sources  : 2.20
flat sources      : 1.70
flat concentration: 81.8% from one branch


In [11]:
detail[['id', 'kind', 'branched_coverage', 'flat_coverage', 'flat_top_branch_share']]

,id,kind,branched_coverage,flat_coverage,flat_top_branch_share
0,r01,single,1.0,1.000,0.833
1,r02,single,1.0,1.000,1.000
2,r03,single,1.0,1.000,1.000
3,r04,single,1.0,1.000,1.000
4,r05,single,1.0,1.000,0.833
5,r06,single,1.0,1.000,0.667
6,r07,single,1.0,1.000,0.917
7,r08,single,1.0,1.000,0.667
8,r09,single,1.0,1.000,1.000
9,n01,single,1.0,1.000,0.833
